In [3]:
!pip install mesa pandas numpy networkx

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)


In [4]:
import mesa
import pandas as pd
import numpy as np

# =====================================================================
# HELPER FUNCTIONS
# =====================================================================
def compute_cgi(model):
    """
    Calculates the Contributions Gini Index (CGI) across all agents.
    A score of 0 means perfect equality (everyone contributes equally).
    A score of 1 means extreme inequality (a tiny core group does all the work).
    """
    contributions = [agent.total_contributed for agent in model.schedule.agents]
    contributions = np.array(contributions, dtype=float)
    
    if np.sum(contributions) == 0:
        return 0.0

    contributions = np.sort(contributions)
    n = len(contributions)
    index = np.arange(1, n + 1)
    
    # Standard Gini coefficient formula
    gini = ((2 * np.sum(index * contributions)) / (n * np.sum(contributions))) - ((n + 1) / n)
    return gini

def compute_free_rider_ratio(model):
    """Calculates the percentage of the ecosystem that is currently free-riding."""
    free_riders = sum(1 for a in model.schedule.agents if a.status == 'free-rider')
    return free_riders / model.num_agents

# =====================================================================
# AGENT DEFINITION (The SME)
# =====================================================================
class SMEAgent(mesa.Agent):
    """An agent representing a Small or Medium Enterprise (SME)."""
    
    def __init__(self, unique_id, model, empirical_data_row):
        super().__init__(unique_id, model)
        
        # ---------------------------------------------------------
        # EMPIRICAL VARIABLES (Passed in from your datasets)
        # ---------------------------------------------------------
        # TODO: Map these to your actual Software Composition Analysis (SCA) 
        # and GitHub dataset columns.
        
        # 1. Firm Size/Revenue (e.g., in Euros)
        self.wealth = empirical_data_row['firm_revenue'] 
        
        # 2. Dependency Level: How many open-source packages they rely on
        self.dependency_count = empirical_data_row['num_dependencies']
        
        # 3. Replacement Cost: COCOMO II estimation of how much value they 
        # are extracting for free.
        self.extracted_value = empirical_data_row['cocomo_value']
        
        # 4. Historical Contribution Rate: Based on GitHub commit logs.
        # Ranges from 0.0 (Pure free-rider) to 1.0 (Heavy contributor)
        self.baseline_propensity = empirical_data_row['github_commit_ratio']
        
        # ---------------------------------------------------------
        # DYNAMIC VARIABLES (Change as the simulation runs)
        # ---------------------------------------------------------
        # Initialize status based on their historical data
        self.status = 'contributor' if self.baseline_propensity > 0.1 else 'free-rider'
        self.total_contributed = self.baseline_propensity * 100 # Arbitrary starting unit
        self.current_capital = self.wealth

    def step(self):
        """
        The core decision-making logic of the SME for each time step.
        The firm weighs the utility of free-riding against contributing.
        """
        
        # --- Option A: Utility of Free-Riding ---
        # They gain the extracted value, but face risks if a "Commons License" is active.
        # The stricter the license, the higher the penalty for extracting without contributing.
        coercion_penalty = self.model.license_strictness * self.dependency_count
        utility_freeride = self.extracted_value - coercion_penalty
        
        # --- Option B: Utility of Contributing ---
        # They pay a cost to contribute (developer time), but gain selective incentives 
        # (institutional funding, reputational benefits, reduced coordination costs).
        cost_to_contribute = self.extracted_value * 0.10 # Assuming contributing costs 10% of replacement value
        utility_contribute = (self.extracted_value - cost_to_contribute) + self.model.selective_incentive
        
        # Add their intrinsic propensity (some firms naturally want to help)
        utility_contribute += (self.baseline_propensity * 5000) 

        # --- Decision Making ---
        if utility_contribute > utility_freeride:
            self.status = 'contributor'
            self.total_contributed += cost_to_contribute
            self.current_capital -= cost_to_contribute
            self.model.total_commons_health += cost_to_contribute # They enrich the commons
        else:
            self.status = 'free-rider'
            self.model.total_commons_health -= (self.extracted_value * 0.01) # Wear and tear / stagnation
            
# =====================================================================
# MODEL DEFINITION (The Digital Economy)
# =====================================================================
class CommonsModel(mesa.Model):
    """A model with empirical SME agents interacting with the digital commons."""
    
    def __init__(self, empirical_dataset, selective_incentive=0, license_strictness=0):
        self.num_agents = len(empirical_dataset)
        self.schedule = mesa.time.RandomActivation(self)
        
        # ---------------------------------------------------------
        # MACRO-ENVIRONMENTAL VARIABLES
        # ---------------------------------------------------------
        # This represents the total "health" or "value" of the digital commons.
        self.total_commons_health = 1000000 
        
        # POLICY INTERVENTIONS (These are what you will tweak for experiments)
        self.selective_incentive = selective_incentive # e.g., Tax breaks, ZenDiS funding
        self.license_strictness = license_strictness   # e.g., Copyleft enforcement, legal risks
        
        # ---------------------------------------------------------
        # AGENT INITIALIZATION (Data Integration)
        # ---------------------------------------------------------
        # Iterate through the Pandas DataFrame to spawn agents based on real data
        for index, row in empirical_dataset.iterrows():
            # Create the agent, passing the row of data
            a = SMEAgent(unique_id=index, model=self, empirical_data_row=row)
            self.schedule.add(a)

        # ---------------------------------------------------------
        # DATA COLLECTION
        # ---------------------------------------------------------
        # We track the variables mentioned in your proposal over time.
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "Commons_Health": "total_commons_health",
                "CGI": compute_cgi,
                "Free_Rider_Ratio": compute_free_rider_ratio
            },
            agent_reporters={"Status": "status", "Capital": "current_capital"}
        )

    def step(self):
        """Advances the model by one time step."""
        self.datacollector.collect(self)
        self.schedule.step()

/home/merijn/projects/cssci/DigitalCommons/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
